# ensemble exp 13 and 23 ด้วย 2 วิธีคือ hard voting and soft voting

# 1.Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)

# 2. Setting

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")

EXP13_DIRS = {
    42: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_seed42",
    43: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_Seed43",
    44: PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_Seed44",
}
EXP23_DIRS = {
    42: PROJECT_DIR / "results" / "CNN_5_7ch_Exp23_seed42",
    43: PROJECT_DIR / "results" / "CNN_5_7ch_Exp23_seed43",
    44: PROJECT_DIR / "results" / "CNN_5_7ch_Exp23_seed44",
}
RESULT_DIR = PROJECT_DIR / "results" / "CNN_5_7ch_Exp13_23_Ensemble"
assert RESULT_DIR not in list(EXP13_DIRS.values()) + list(EXP23_DIRS.values())
RESULT_DIR.mkdir(parents=True, exist_ok=True)

STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
PROB_COLS = ["prob_" + s for s in STAGES]
KEY_COLS = ["subject", "epoch_start", "epoch_end", "actual"]
SEEDS = [42, 43, 44]

# 3.โหลด CSV และตรวจความตรงกัน

In [ ]:
MODELS = {f"Exp13_s{s}": EXP13_DIRS[s] for s in SEEDS}
MODELS.update({f"Exp23_s{s}": EXP23_DIRS[s] for s in SEEDS})

saved = {n: pd.read_csv(d / "val_predictions.csv") for n, d in MODELS.items()}
base = saved["Exp13_s42"]
assert len(base) == 2878

for name, df in saved.items():
    assert len(df) == len(base), name
    for col in KEY_COLS:
        assert (df[col].astype(str).to_numpy() == base[col].astype(str).to_numpy()).all(), f"{name}: {col} ไม่ตรงกัน"
    prob = df[PROB_COLS].to_numpy()
    assert np.isfinite(prob).all()
    assert np.allclose(prob.sum(axis=1), 1, atol=1e-3)
    assert (np.array(STAGES)[prob.argmax(axis=1)] == df["predicted"].to_numpy()).all(), name

y_val = base["actual"].map(dict(zip(STAGES, LABELS))).to_numpy().astype(int)
probs = {n: df[PROB_COLS].to_numpy() for n, df in saved.items()}
print("OK:", len(y_val), "sequences,", base["subject"].nunique(), "subjects")

# 4.Hard Voting และ Soft Voting

In [ ]:
def hard_vote(names, tie_mode):
    P = np.stack([probs[n] for n in names])
    votes = P.argmax(axis=2)
    counts = np.stack([(votes == k).sum(axis=0) for k in LABELS], axis=1)
    tied = counts == counts.max(axis=1, keepdims=True)
    is_tie = tied.sum(axis=1) > 1
    pred = counts.argmax(axis=1)
    if tie_mode == "first":
        ref = votes[0]
    else:
        ref = np.where(tied, P.mean(axis=0), -1).argmax(axis=1)
    pred[is_tie] = ref[is_tie]
    return pred, is_tie

def soft_vote(names):
    return np.mean([probs[n] for n in names], axis=0).argmax(axis=1)

e13 = [f"Exp13_s{s}" for s in SEEDS]
e23 = [f"Exp23_s{s}" for s in SEEDS]
GROUPS = {"Exp13": (e13, "first"), "Exp23": (e23, "first"), "Mix6": (e13 + e23, "prob")}

preds = {n: probs[n].argmax(axis=1) for n in MODELS}
for g, (names, tm) in GROUPS.items():
    preds[f"{g}_hard"], tie = hard_vote(names, tm)
    preds[f"{g}_soft"] = soft_vote(names)
    print(g, "hard เสมอ", int(tie.sum()), "ตัวอย่าง")

# 5.คำนวณ metric แสดงผล เปรียบเทียบ และบันทึก

In [ ]:
def calc_metrics(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    overall = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision.mean(),
        "macro_recall": recall.mean(),
        "macro_f1": f1.mean(),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred, labels=LABELS),
    }
    per_class = pd.DataFrame({
        "stage": STAGES, "precision": precision, "recall": recall,
        "f1_score": f1, "support": support,
    })
    return overall, per_class


def stage_confusion(cm):
    rows = []
    total = cm.sum()
    for i, stage in enumerate(STAGES):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - tp - fp - fn
        rows.append({
            "stage": stage, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "sensitivity": tp / (tp + fn) if tp + fn > 0 else 0,
            "specificity": tn / (tn + fp) if tn + fp > 0 else 0,
        })
    return pd.DataFrame(rows)


def show_result(name, y_pred):
    overall, per_class = calc_metrics(y_val, y_pred)
    cm = confusion_matrix(y_val, y_pred, labels=LABELS)
    cm_norm = confusion_matrix(y_val, y_pred, labels=LABELS, normalize="true")

    print("=====", name, "(VAL) =====")
    display(pd.DataFrame([overall]).round(4))
    display(per_class.round(4))
    display(stage_confusion(cm).round(4))
    print("Confusion matrix (จำนวน, แถว=จริง, คอลัมน์=ทาย)")
    display(pd.DataFrame(cm, index=[f"จริง {s}" for s in STAGES],
                         columns=[f"ทาย {s}" for s in STAGES]))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    ConfusionMatrixDisplay(cm, display_labels=STAGES).plot(ax=axes[0], cmap="Blues", colorbar=False)
    axes[0].set_title("Confusion matrix (count)")
    ConfusionMatrixDisplay(cm_norm, display_labels=STAGES).plot(ax=axes[1], cmap="Blues", values_format=".2f", colorbar=False)
    axes[1].set_title("Confusion matrix (normalized by actual)")
    plt.suptitle(name)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{name}_val_confusion_matrix.png", dpi=150)
    plt.show()

    pd.DataFrame([overall]).to_csv(RESULT_DIR / f"{name}_val_overall_metrics.csv", index=False)
    per_class.to_csv(RESULT_DIR / f"{name}_val_per_class_metrics.csv", index=False)
    stage_confusion(cm).to_csv(RESULT_DIR / f"{name}_val_per_stage_confusion.csv", index=False)
    pd.DataFrame(cm, index=STAGES, columns=STAGES).to_csv(RESULT_DIR / f"{name}_val_confusion_matrix.csv")
    return overall, per_class


rows, f1_rows = {}, {}
for name, p in preds.items():
    overall, pc = calc_metrics(y_val, p)
    rows[name], f1_rows[name] = overall, pc["f1_score"].to_numpy()

comparison = pd.DataFrame(rows).T
comparison.index.name = "method"
f1_comparison = pd.DataFrame(f1_rows, index=STAGES).T
f1_comparison.index.name = "method"

comparison.to_csv(RESULT_DIR / "val_comparison_overall.csv")
f1_comparison.to_csv(RESULT_DIR / "val_comparison_f1_per_class.csv")

pred_table = base[["subject", "epoch_start", "epoch_end", "actual"]].copy()
for name, p in preds.items():
    pred_table[f"pred_{name}"] = np.array(STAGES)[p]
pred_table.to_csv(RESULT_DIR / "val_ensemble_predictions.csv", index=False)


KEY = ["Exp13_hard", "Exp23_hard", "Mix6_hard",
       "Exp13_soft", "Exp23_soft", "Mix6_soft"]

display(comparison.loc[["Exp13_hard", "Exp13_soft"],
                       ["accuracy", "macro_f1", "cohen_kappa"]].round(4))


t1 = comparison.loc[KEY, ["accuracy", "macro_f1", "cohen_kappa"]].copy()
for m in ["macro_f1", "cohen_kappa"]:
    ref = [comparison.loc["Exp13_" + k.split("_")[1], m] for k in KEY]
    t1[f"Δ{m} vs Exp13"] = t1[m].to_numpy() - np.array(ref)
print("ตาราง 1: ภาพรวม (Δ เทียบ Exp13 voting แบบเดียวกัน)")
display(t1.round(4))
t1.round(4).to_csv(RESULT_DIR / "summary_table.csv")


print("ตาราง 2: F1 รายคลาส")
display(f1_comparison.loc[KEY].round(4))


print("ตาราง 3: Mix6 − Exp13 (F1 รายคลาส)")
t3 = pd.DataFrame({
    v: f1_comparison.loc[f"Mix6_{v}"] - f1_comparison.loc[f"Exp13_{v}"]
    for v in ["hard", "soft"]
}).T
display(t3.round(4))


print("ตาราง 4: โมเดลเดี่ยว")
display(comparison.loc[list(MODELS), ["accuracy", "macro_f1", "cohen_kappa"]].round(4))


for name in KEY:
    show_result(name, preds[name])

In [ ]:
cm13 = confusion_matrix(y_val, preds["Exp13_hard"], labels=LABELS)
cm23 = confusion_matrix(y_val, preds["Exp23_soft"], labels=LABELS)

idx = [f"จริง {s}" for s in STAGES]
col = [f"ทาย {s}" for s in STAGES]
diff = pd.DataFrame(cm23 - cm13, index=idx, columns=col)
print("Exp23_soft − Exp13_hard (แถว=จริง, คอลัมน์=ทาย)")
display(diff)
diff.to_csv(RESULT_DIR / "val_cm_diff_Exp23soft_vs_Exp13hard.csv")

def pair(cm):
    return {
        "N2→N3": cm[2, 3], "N3→N2": cm[3, 2],
        "R→N2": cm[4, 2], "R→N3": cm[4, 3],
        "ทำนายเป็น N1 ผิด": cm[:, 1].sum() - cm[1, 1],
        "N1 ถูก": cm[1, 1],
        "ผิดทั้งหมด": cm.sum() - np.trace(cm),
    }

pairs = pd.DataFrame({"Exp13_hard": pair(cm13), "Exp23_soft": pair(cm23)})
pairs["ผลต่าง"] = pairs["Exp23_soft"] - pairs["Exp13_hard"]
display(pairs)
pairs.to_csv(RESULT_DIR / "val_error_pairs_Exp13hard_vs_Exp23soft.csv")